# 04 · Validate — pose maintenance, developability liability scan, epistasis

**Standard slot:** *validate (in silico).* **For Project 15 the core analyses are:** (1) **pose
maintenance** — which candidates keep the parent binding pose (AF2-Multimer `pae_interaction` + scRMSD
vs parent), (2) the **developability liability scan** (flag NG/DG deamidation, Met oxidation, unpaired
Cys, glyc sequons introduced into the CDRs), and (3) **epistasis / combination** reasoning — do
single-mutation gains add up, or interfere? (D3 pt 2).

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC ranking values** — the
figures demonstrate the analysis; real numbers come from ESM-1v/AbLang + a batched AF2-Multimer run.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Pose maintenance — does the mutation keep the binding pose? `[core]`

The decisive maturation filter is not "is the ESM-1v score high" but "does the antibody still dock the
antigen the same way". Plot each candidate's `pae_interaction` (interface confidence; want ≤ 12) against
its `scrmsd` vs the parent pose (want ≤ 3.0). The good quadrant is **low-pae, low-scrmsd**. A high
ESM-1v rank with a broken pose is a false lead — this is exactly the cross-check a single-sequence
language model cannot do on its own.

In [ ]:
import pandas as pd
from maturation_tools import (example_parent_sequence, apply_mutation, Variant, score_variants)

camp = pd.read_csv("results/campaign.csv")
parent = example_parent_sequence()

# Rebuild + pose-check the top single-mutation candidates (by ESM-1v) for the figure.
top_single = camp[camp["source"] == "esm1v"].sort_values("esm1v", ascending=False).head(20)
vs = []
for _, r in top_single.iterrows():
    seq = parent
    try:
        for m in str(r["mutations"]).split("+"):
            if m:
                seq = apply_mutation(seq, m)
    except Exception:
        continue
    vs.append(Variant(design_id=str(r["design_id"]), sequence=seq, mutations=tuple(str(r["mutations"]).split("+")),
                      cdr=str(r["cdr"]), esm1v=r.get("esm1v"), ablang=r.get("ablang")))
score_variants(vs, tool="mock")
pose_df = pd.DataFrame([dict(design_id=v.design_id, mutation="+".join(v.mutations), cdr=v.cdr,
                            esm1v=v.esm1v, pae_interaction=v.pae_interaction, scrmsd=v.scrmsd,
                            n_liabilities=v.n_liabilities) for v in vs])
print("pose-maintenance table (SYNTHETIC mock metrics):")
pose_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4))
sc = ax.scatter(pose_df["scrmsd"], pose_df["pae_interaction"], c=pose_df["esm1v"], cmap="viridis", s=40)
ax.axvline(3.0, color="r", ls="--", lw=1, label="scrmsd cutoff 3.0")
ax.axhline(12, color="orange", ls="--", lw=1, label="pae cutoff 12")
ax.set_xlabel("scRMSD vs parent pose (Å)"); ax.set_ylabel("pae_interaction (Å)")
ax.set_title("Pose maintenance — SYNTHETIC (want low-left quadrant)")
plt.colorbar(sc, label="ESM-1v Δ-LL"); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("results/pose_maintenance.png", dpi=150); plt.show()
print("Good candidates: maintain the pose (low pae + low scrmsd) AND rank well on ESM-1v.")

## 2 · Developability liability scan `[core]`

A maturation mutation can quietly introduce a chemical liability into a CDR: an **NG/NS** Asn
**deamidation** hotspot, a **DG/DS** Asp **isomerization** hotspot, a solvent-exposed **Met** (oxidation)
or **Trp**, an **N-glycosylation sequon**, or an **unpaired Cys** (disulfide scrambling). These age the
drug, hurt manufacturability, and can sit right in the paratope. `developability_scan()` flags these
motifs **inside the CDRs**. It is a **TEACHING HEURISTIC**, not real TAP / a structure-based deamidation
predictor — use it to triage obvious liabilities **before** synthesis; confirm with the real tools for
any claim.

In [ ]:
from maturation_tools import developability_scan

# Scan the parent vs each top candidate; flag candidates that ADD a liability the parent didn't have.
parent_scan = developability_scan(parent)
parent_liab = set(parent_scan["liabilities"])
print("parent CDR liabilities (heuristic):", parent_scan["n_liabilities"], parent_scan["liabilities"])

rows = []
for v in vs:
    scan = developability_scan(v.sequence)
    introduced = set(scan["liabilities"]) - parent_liab
    rows.append(dict(design_id=v.design_id, mutation="+".join(v.mutations),
                     total_liabilities=scan["n_liabilities"],
                     introduced=";".join(f"{m}@{p}" for m, p in sorted(introduced)) or "(none)"))
liab_df = pd.DataFrame(rows)
print("\ncandidates that INTRODUCE a new CDR liability (avoid these):")
liab_df[liab_df["introduced"] != "(none)"].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(liab_df["total_liabilities"], bins=range(0, max(liab_df["total_liabilities"].max(), 1) + 2))
ax.set_xlabel("CDR liability motif count (heuristic)"); ax.set_ylabel("candidates")
ax.set_title("Developability liabilities per candidate (NOT real TAP)")
plt.tight_layout(); plt.savefig("results/developability.png", dpi=150); plt.show()
print("Prefer candidates that ADD no liability. Confirm with real TAP / deamidation tools before any claim.")

## 3 · Epistasis / combination reasoning `[extension]`

Single mutations are scored independently, but the bench tests *combinations*. **Epistasis** means the
effect of two mutations together is not the sum of their separate effects — they can reinforce or
interfere, especially when close in 3-D. You cannot get a real ΔΔG from the mock backend, so the
deliverable is the **reasoning + the plan**: take the few best single mutations, propose pairwise
combinations, re-score the pose (a real combo can break the interface even if both singles maintain it),
and flag pairs to test as a small block. Combinations to actually order stay FEW — the validation
budget is small.

In [ ]:
import itertools
from maturation_tools import assemble_candidate_set

# Take the best few single mutations (distinct positions) and propose pairwise combinations.
best_singles = assemble_candidate_set([v for v in vs], top_n=4)
combos = []
for a, b in itertools.combinations(best_singles, 2):
    ma, mb = a.mutations[0], b.mutations[0]
    # apply both to the parent (skip if they hit the same position)
    pa = int(ma[1:-1]); pb = int(mb[1:-1])
    if pa == pb:
        continue
    seq = parent
    try:
        seq = apply_mutation(apply_mutation(parent, ma), mb)
    except Exception:
        continue
    combos.append(Variant(design_id=f"EXAMPLE_DATA_COMBO_{ma}_{mb}", sequence=seq,
                          mutations=(ma, mb), source="combination"))
score_variants(combos, tool="mock")
combo_df = pd.DataFrame([dict(mutations="+".join(c.mutations), pae_interaction=c.pae_interaction,
                             scrmsd=c.scrmsd, n_liabilities=c.n_liabilities) for c in combos])
print("pairwise combinations to TEST (SYNTHETIC pose metrics; epistasis confirmed only by SPR):")
combo_df

## D3 (part 2) checklist
- [ ] Pose-maintenance figure (pae_interaction vs scRMSD vs parent) + the good-quadrant call.
- [ ] Developability liability scan: candidates that **introduce** a CDR liability flagged + dropped.
- [ ] Epistasis/combination reasoning: a FEW pairwise combos proposed + pose-rechecked; plan to test.
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as ranking/pose/plumbing, not affinity.

**Next:** `05_validation_plan.ipynb` — the SPR/DSF validation plan with controls.